# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides an example for loading and exploring a FAIR<sup>2</sup> dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. The notebook demonstrates key steps for metadata inspection, record/field overview, data extraction, EDA, and basic visualization, referencing all dataset entities by their `@id` as required for Croissant compliance.

### Dataset Source
The dataset source is defined by a [Croissant schema](https://mlcommons.org/croissant/) at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\n\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs using the Croissant schema.

We will list the record sets, their `@id`s, and the available fields/columns for each one.

In [ ]:
# Show all record set @ids and their fields (by @id)
record_sets = []
for rs in getattr(metadata, 'recordSet', []):
    # Each rs is an mlcroissant.RecordSet object
    print(f"RecordSet: {rs['@id']} (name: {getattr(rs, 'name', '[no name]')})")
    if hasattr(rs, 'field'):
        fields = rs.field if isinstance(rs.field, list) else [rs.field]
        print('  Fields:')
        for fld in fields:
            print(f"    - {fld['@id']}")
    elif hasattr(rs, 'column'):
        columns = rs.column if isinstance(rs.column, list) else [rs.column]
        print('  Columns:')
        for col in columns:
            print(f"    - {col['@id']}")
    print()
    record_sets.append(rs['@id'])
if not record_sets:
    print('No record sets found in metadata. If loading fails, check dataset schema.')

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. **All record set and field identifiers use their `@id`**. 

We'll dynamically gather all record set ids and attempt to load their data.

In [ ]:
# Extract data from each record set using their @id
if len(record_sets) == 0:
    print('No record sets available to extract.')
else:
    dataframes = {}
    for record_set_id in record_sets:
        try:
            records = list(dataset.records(record_set=record_set_id))
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f'RecordSet {record_set_id} loaded. Shape: {df.shape}')
            print(f'Columns: {df.columns.tolist()}\n')
        except Exception as e:
            print(f"Failed to load {record_set_id}: {e}\n")
    # For demonstration, pick the first record set with records
    example_set = None
    for k, v in dataframes.items():
        if not v.empty:
            example_set = k
            break
    if example_set:
        print(f"First rows from {example_set}:")
        display(dataframes[example_set].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing and cleaning methods:
- Filtering records based on numeric thresholds
- Normalizing columns
- Grouping by categorical field
- All field references are by `@id`

In [ ]:
# Pick record set and numeric/categorical fields by @id
if len(dataframes) > 0 and example_set:
    df = dataframes[example_set]
    # Try to guess a numeric field by pandas dtype
    numeric_candidates = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    if not numeric_candidates:
        # Try to coerce columns if all are strings (common for Croissant tabular)
        for c in df.columns:
            # Infer numeric-looking columns
            try:
                df[c+'_tmp'] = pd.to_numeric(df[c], errors='coerce')
                if df[c+'_tmp'].notnull().any():
                    numeric_candidates.append(c)
                df.drop(columns=[c+'_tmp'], inplace=True)
            except Exception:
                continue
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
        print(f"Using numeric field for EDA: {numeric_field}")
        # Try thresholding at a low quantile
        ser = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = np.nanpercentile(ser.dropna(), 70) if ser.notnull().any() else 10
        filtered_df = df[ser > threshold].copy()
        print(f"Filtered records: {filtered_df.shape[0]}")
        filtered_df[f"{numeric_field}_normalized"] = (ser - ser.mean()) / ser.std()
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Try grouping by another field
        group_field = None
        for c in df.columns:
            if c != numeric_field and (df[c].nunique() < 8):
                group_field = c
                break
        if group_field:
            print(f"Grouping by field: {group_field}")
            grouped = filtered_df.groupby(group_field).agg({numeric_field:['mean','count']})
            display(grouped)
        else:
            print('No suitable categorical field for grouping found.')
    else:
        print('No numeric field found for analysis. Please inspect dataset.')
else:
    print('No data available for EDA in the record sets.')

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0 and example_set and 'numeric_field' in locals():
    ser = pd.to_numeric(dataframes[example_set][numeric_field], errors='coerce')
    plt.figure(figsize=(8,4))
    sns.histplot(ser.dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If group_field exists, boxplot
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=dataframes[example_set][group_field], y=ser)
        plt.title(f"{numeric_field} by {group_field}")
        plt.ylabel(numeric_field)
        plt.xlabel(group_field)
        plt.show()
else:
    print('No numeric field or group field found to visualize.')

## 6. Conclusion
We successfully loaded and inspected the FAIR<sup>2</sup> clinical dataset using Croissant and `mlcroissant`. All manipulations referenced schema entities by their `@id` as required. 
- Use this notebook as a template for referencing, loading, analyzing, and plotting Croissant datasets in your own projects.